In [ ]:
import sys
sys.path.append(os.path.abspath(".."))

In [ ]:
from conifg import Cofig as cfg
import json
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

# **Filtering datas having problems with format**

In [ ]:
with open(cfg.DUMMY_DATA_PATH, 'r') as f:
  data = json.load(f)

In [ ]:
dataset_for_filtering = Dataset.from_dict(data)

In [ ]:
dataset_for_filtering

Dataset({
    features: ['original_article', 'has_subject', 'has_keywords', 'has_summary', 'num_summary_tokens', 'subject', 'summary', 'keywords', 'model_output'],
    num_rows: 30
})

In [ ]:
#Temperature=0.7, TopP=0.8, TopK=20, and MinP=0 -> recommended config in non-thinking mode by the supplier
from transformers import GenerationConfig
generation_config = GenerationConfig(
    temperature = 0.7,
    top_p = 0.8,
    top_k = 20,
    min_p = 0,
    max_new_tokens = 500)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'min_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
def model_output(tokenizer, message):
  # A function model generates
  #Qwen3 4b model has thinking part so have to remove thinking parts when we save model generation
  #message -> chat templated -> tokenize -> generate
    chat_template_data = tokenizer.apply_chat_template(message, tokenize = False, truncation = True, enable_thinking = False, add_generation_prompt = True)
    model_inputs = tokenizer([chat_template_data], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            generation_config=generation_config,
        )
    output_ids = generated_ids[0].cpu().tolist()
    # remove prompt
    index = len(output_ids) - output_ids[::-1].index(151668)

    # decode
    content = tokenizer.decode(
        generated_ids[0][index:].cpu(),
        skip_special_tokens=True
    ).strip()

    return content

import re


def validate_data(data, tokenizer):

    model_output = data

    # --------------------
    # Subject
    # --------------------
    subject_match = re.search(
        r"subject\s*:\s*(.*?)(?:\n|keywords\s*:|summary\s*:|$)",
        data,
        re.I | re.S
    )

    if subject_match:
        subject = subject_match.group(1).strip()
        has_subject = 1
    else:
        subject = ""
        has_subject = 0

    # --------------------
    # Keywords
    # --------------------
    keywords_match = re.search(
        r"keywords\s*:\s*(.*?)(?:\n|summary\s*:|$)",
        data,
        re.I | re.S
    )

    if keywords_match:
        keywords = keywords_match.group(1).strip()
        has_keywords = 1
    else:
        keywords = ""
        has_keywords = 0

    # --------------------
    # Summary
    # --------------------
    summary_match = re.search(
        r"summary\s*:\s*(.*)",
        data,
        re.I | re.S
    )

    if summary_match:
        summary = summary_match.group(1).strip()
        has_summary = 1
        num_summary_tokens = len(tokenizer.encode(summary))
    else:
        summary = ""
        has_summary = 0
        num_summary_tokens = 0

    return [
        model_output,
        has_subject,
        has_keywords,
        has_summary,
        num_summary_tokens,
        subject,
        keywords,
        summary
    ]

In [ ]:
#load model for re-generating
model_name = cfg.BASE_MODEL_NAME

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
#Now have to check the data is reliable or not.
#by sentence similarity model google/embeddinggemma-300m
#extract 10 percents of lower similarity between original article and summary. -> Maybe weak or bad summary
#10 percents of high similarity -> maybe dublicate original
#10 percents of random among rest of -> can check in general
#after that by using second best model in summary (in my test)

In [ ]:
# from datasets import Dataset
# dataset_for_filtering =  Dataset.from_dict(dataset_dict)

In [ ]:
# checking generated datas
container_for_no_subject = []
container_for_no_keywords = []
container_for_no_summary = []
container_for_short_summary = []
container_for_long_summary = []
container_for_less_keywords = []

for i, data in enumerate(dataset_for_filtering):
  if data['has_subject'] == 0:
    container_for_no_subject.append(i)
  if data['has_keywords'] == 0:
    container_for_no_keywords.append(i)
  if data['has_summary'] == 0:
    container_for_no_summary.append(i)
  if data['num_summary_tokens'] < 30:
    container_for_short_summary.append(i)
  if data['num_summary_tokens'] > 200:
     container_for_long_summary.append(i)

  keywords = [k.strip() for k in data['keywords'].split(",") if k.strip()]
  count = len(keywords)
  if count < 3:
    container_for_less_keywords.append(i)



In [ ]:
#checking data numbers
print('data with no subject:' ,container_for_no_subject)
print('data with no keywords:' ,container_for_no_keywords)
print('data with no summary:' ,container_for_no_summary)
print('data with short summary:' ,set(container_for_short_summary)-set(container_for_no_summary))
print('data with long summary:' ,container_for_long_summary)
print('data with less than 3 keywords:' ,container_for_less_keywords)

data with no subject: [10, 11, 12, 13, 14]
data with no keywords: [15, 16, 17, 18, 19]
data with no summary: [20, 21, 22]
data with short summary: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 23, 24, 25, 26, 27, 28, 29}
data with long summary: []
data with less than 3 keywords: [15, 16, 17, 18, 19, 26, 27]


In [ ]:
# Some datas have problems with Summary.
datas_with_summary_problems = list(set(container_for_short_summary + container_for_long_summary) -set(container_for_no_summary))
for i in datas_with_summary_problems:
  print(dataset_for_filtering[i]['model_output'], i)
  print(' ')
  user_input = input("Continue to next chunk? (y/n): ")
  if user_input.lower() != "y":
    print(' ')
    break

Subject: Company 0
Keywords: Company0, product, release
Summary: Company 0 released a new product and gained attention. 0
 
Continue to next chunk? (y/n): y
Subject: Company 1
Keywords: Company1, product, release
Summary: Company 1 released a new product and gained attention. 1
 
Continue to next chunk? (y/n): y
Subject: Company 2
Keywords: Company2, product, release
Summary: Company 2 released a new product and gained attention. 2
 
Continue to next chunk? (y/n): n
 


In [ ]:
data_with_problems =list(set(container_for_no_subject + container_for_no_keywords + container_for_no_summary + container_for_short_summary + container_for_long_summary + container_for_less_keywords))

**regenerating datas with problems with format**

In [ ]:
# re-generate data without one shot
dict_for_gen = {'original_article': [] ,'model_output' : [] ,'has_subject': [], 'has_keywords': [] , 'has_summary': [] , 'num_summary_tokens': [],'subject': [] ,'keywords': [], 'summary':[], 'data_num' : []}
for i in data_with_problems:


  message = [{'role': 'system', 'content' : 'Your are going to summarize text. it has to include "Subject:", "Keywords:" and "Summary:" parts. The number of Keywords must be less than 5'},
    {'role' :'user' , 'content' : 'Summarize the follwing text:  ' + dataset_for_filtering['original_article'][i]}]

  output = model_output(tokenizer,message)
  features = validate_data(output, tokenizer)

  dict_for_gen['original_article'].append(dataset_for_filtering['original_article'][i])
  dict_for_gen['model_output'].append(output)
  dict_for_gen['has_subject'].append(features[1])
  dict_for_gen['has_keywords'].append(features[2])
  dict_for_gen['has_summary'].append(features[3])
  dict_for_gen['num_summary_tokens'].append(features[4])
  dict_for_gen['keywords'].append(features[5])
  dict_for_gen['summary'].append(features[6])
  dict_for_gen['data_num'].append(i)



In [ ]:
dataset_for_problems = Dataset.from_dict(dict_for_gen)

In [ ]:
from datasets import Dataset

def patch_dataset(original_dataset, fixed_dataset):
    """
    original_dataset: dataset with original data
    fixed_dataset: dataset with fixed data
    return: patched dataset
    """

    original_dict = original_dataset.to_dict()

    columns = original_dataset.column_names

    for i in range(len(fixed_dataset)):
        fixed_row = fixed_dataset[i]
        idx = fixed_row["data_num"]

        # overwrite
        for col in columns:
            if col in fixed_row:
                original_dict[col][idx] = fixed_row[col]

    # turning into dataset
    patched_dataset = Dataset.from_dict(original_dict)

    return patched_dataset

In [ ]:
preprocessed_dataset = patch_dataset(dataset_for_filtering, dataset_for_problems)

In [ ]:
#save pre-processed dataset
with open('dataset_preprocessed.json','w') as f:
  json.dump(Dataset.to_dict(preprocessed_dataset), f)

# **Sampling with Similarity model starts**

In [ ]:
#load preprocessed data
with open('dataset_preprocessed.json','r') as f:
  data = json.load(f)

In [ ]:
data = dataset_for_filtering

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
model = SentenceTransformer(cfg.SENTENCE_MODEL)


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

In [ ]:
data = Dataset.from_dict(data)

In [ ]:
container_for_similarity = []
for i, data_ in enumerate(data):
  query = data['summary'][i]
  documents = [data['original_article'][i]]
  query_embeddings = model.encode_query(query)
  document_embeddings = model.encode_document(documents)
  #print(query_embeddings.shape, document_embeddings.shape)
  # (768,) (4, 768)

  # Compute similarities to determine a ranking
  similarities = model.similarity(query_embeddings, document_embeddings)
  #print(similarities)
  container_for_similarity.append(similarities)
  # tensor([[0.3011, 0.6359, 0.4930, 0.4889]])

In [ ]:
scores_tensor = torch.tensor([s.item() for s in container_for_similarity])

In [ ]:
# 10% of dataset
n =  int((len(data['original_article'])/100)*10)

values_low, indices_low = torch.topk(scores_tensor, k=n, largest=False)

print(indices_low)

tensor([12, 28, 29])


In [ ]:
n =  int((len(data['original_article'])/100)*10)

values_high, indices_high = torch.topk(scores_tensor, k=n, largest=True)

print(indices_high)

tensor([3, 5, 6])


In [ ]:
#pick random datas not in high and low similarity list

In [ ]:
pick_num = [ i for i in range(len(data['model_output']))]

In [ ]:
num = (set(pick_num) - set(indices_low.tolist()))- set(indices_high.tolist())

In [ ]:
import random
random_num = random.sample(list(num), int((len(data['original_article'])/100)*10))

In [ ]:
sampled_num = indices_low.tolist() + indices_high.tolist() + random_num

In [ ]:
#save data-list for llm judge
with open('sampled_ids.json', 'w') as f:
  json.dump( {'num_high' : indices_high.tolist() , 'nums_low': indices_low.tolist() , 'nums_random': random_num}, f)

In [ ]:
with open('sampled_ids.json', 'r' ) as f:
  sampled_ids = json.load(f)

In [ ]:
# now Second best model in summary test would evaluate quality of picked datas
# model gemma-3-4b-it

# **Model Evaluation Filtering with sampled data starts**

In [ ]:
# now Second best model in summary test would evaluate quality of picked datas
# model gemma-3-4b-it
#gemma model asks for huugging face key
from huggingface_hub import login
login()

In [ ]:
from transformers import pipeline
import torch
model_name = cfg.EVAL_MODEL
pipe = pipeline(
    "image-text-to-text",
    model=model_name,
    device="cuda",
    torch_dtype=torch.bfloat16
)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [ ]:
system = ''' You are an expert evaluator of text summarization quality.

Your task is to evaluate whether a generated summary is reliable and usable for training purposes.

Focus only on serious issues. Minor wording differences or stylistic variations are acceptable.

Evaluate the summary according to the following criteria:

1. Faithfulness:
   - Does the summary contain any factual information that is not supported by the original article?
   - Does it hallucinate events, numbers, names, or claims?

2. Contradiction:
   - Does the summary contradict any information in the article?

3. Relevance:
   - Is the summary clearly about the same topic as the article?
   - Is it mostly aligned with the article’s main content?

Ignore minor omissions, compression artifacts, or stylistic simplifications.
Overall Reliability score scales 0 to 100
Output format:

Hallucination: Yes/No
Contradiction: Yes/No
Irrelevant: Yes/No
Overall Reliability Score (0–100): <number>
Short Explanation: <1–3 sentences explaining the decision>'''

In [ ]:
import re

def eval_validate(text):
    result = {
        "has_Halucination": 0,
        "has_Contradiction": 0,
        "has_Irrelevant": 0,
        "has_Overall_Reliability": None,
        "has_Short_Explanation": 0,
    }

    # Hallucination
    match = re.search(r"Hallucination:\s*(Yes|No)", text, re.IGNORECASE)
    if match:
        result["has_Halucination"] = 1 if match.group(1).lower() == "yes" else 0

    # Contradiction
    match = re.search(r"Contradiction:\s*(Yes|No)", text, re.IGNORECASE)
    if match:
        result["has_Contradiction"] = 1 if match.group(1).lower() == "yes" else 0

    # Irrelevant
    match = re.search(r"Irrelevant:\s*(Yes|No)", text, re.IGNORECASE)
    if match:
        result["has_Irrelevant"] = 1 if match.group(1).lower() == "yes" else 0

    # 🔥 Improved Overall Reliability Score regex
    match = re.search(
        r"Overall\s+Reliability\s+Score(?:\s*\(.*?\))?\s*:\s*(\d+)",
        text,
        re.IGNORECASE
    )
    if match:
        result["has_Overall_Reliability"] = int(match.group(1))

    # Short Explanation existence
    if re.search(r"Short\s+Explanation\s*:", text, re.IGNORECASE):
        result["has_Short_Explanation"] = 1

    return result['has_Halucination'], result['has_Contradiction'], result['has_Irrelevant'], result['has_Overall_Reliability'], result['has_Short_Explanation']

In [ ]:
from tqdm import tqdm

In [ ]:
def gemma3_eval(data, id_list, system, group="no"):
  eval = {'Evaluation' : [] , 'has_Halucination' : [] , 'has_Contradiction' : [] ,'has_Irrelevant' : [] , 'has_Overall_Reliability' : [] , 'has_Short_Explanation' : [] , 'ID' : [], 'Group' : group }
  for id in tqdm(id_list, desc = 'Eval'):
    article = data[id]['original_article']
    summary = data[id]['summmary']
    messages = [
      {
          "role": "system",
          "content": [{"type": "text", "text": system}]
      },
      {
          "role": "user",
          "content": [
              #{"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
              {"type": "text", "text": 'original: '  +article + '\n' + 'Summary: ' +  summary}
          ]
        }
     ]

    output = pipe(text=messages, max_new_tokens=200)
    result = output[0]["generated_text"][-1]["content"]
    evaluation = result
    halucination, contradiction, irrelevant, overall_reliability, short_explanation =  eval_validate(result)
    eval['Evaluation'].append(evaluation)
    eval['has_Halucination'].append(halucination)
    eval['has_Contradiction'].append(contradiction)
    eval['has_Irrelevant'].append(irrelevant)
    eval['has_Overall_Reliability'].append(overall_reliability)
    eval['has_Short_Explanation'].append(short_explanation)
    eval['ID'].append(id)
    print('ID : {}, has_halu: {}, has_contra: {}, has_irrelevant:{}, has_reliability:{}, has_short: {}'.format(id,halucination, contradiction, irrelevant, overall_reliability, short_explanation))
    # print(evaluation)
    # stop = input()
    # if stop == 'stop':
    #   break
  return eval


In [ ]:
evaluation = gemma3_eval(data, sampled_ids['num_high'], system, group = 'High')

In [ ]:
#save evaluation for high similrarity datas
with open('evaluation_high.json','w') as f:
  json.dump(evaluation, f)

In [ ]:
# @title
evaluation_low = gemma3_eval(data, sampled_ids['nums_low'], system, group = 'Low')

In [ ]:
#save evaluation for low similrarity datas
with open('evaluation_low.json','w') as f:
  json.dump(evaluation_low, f)

In [ ]:
evaluation = gemma3_eval(data, sampled_ids['nums_random'], system, group = 'Random')

In [ ]:
#save evaluation for random similrarity datas
with open('evaluation_random.json','w') as f:
  json.dump(evaluation, f)

In [ ]:
container_for_id_score = []
for i in range(len(evaluation['ID'])):
  container_for_id_score.append((evaluation['ID'][i], evaluation['has_Overall_Reliability'][i], i))


In [ ]:
print(container_for_id_score)

[(899, 95, 0), (631, 95, 1), (447, 95, 2), (407, 95, 3), (351, 95, 4), (101, 95, 5), (139, 95, 6), (21, 95, 7), (87, 95, 8), (242, 95, 9), (672, 95, 10), (246, 95, 11), (269, 95, 12), (889, 95, 13), (461, 95, 14), (460, None, 15), (158, 95, 16), (476, 100, 17), (511, 95, 18), (548, 95, 19), (907, 100, 20), (930, 95, 21), (909, 95, 22), (831, 95, 23), (81, 95, 24), (597, 95, 25), (862, 95, 26), (543, 95, 27), (514, 98, 28), (579, 95, 29), (350, 95, 30), (811, 95, 31), (917, 95, 32), (879, 100, 33), (495, 95, 34), (284, 95, 35), (331, 95, 36), (51, 95, 37), (292, 100, 38), (425, 95, 39), (103, 95, 40), (552, 95, 41), (822, 98, 42), (174, 95, 43), (83, 95, 44), (630, 98, 45), (937, 95, 46), (29, 95, 47), (723, 95, 48), (381, 95, 49), (800, 95, 50), (932, 95, 51), (465, 95, 52), (148, 75, 53), (947, 95, 54), (69, 95, 55), (855, 95, 56), (279, 95, 57), (509, 95, 58), (276, 95, 59), (398, 95, 60), (581, 95, 61), (487, 95, 62), (324, None, 63), (71, 98, 64), (429, 95, 65), (871, 95, 66), (727

In [ ]:
#load evaluations
with open('evaluation_high.json','r') as f:
  evaluation_high = json.load(f)
with open('evaluation_low.json','r') as f:
  evaluation_low = json.load(f)
with open('evaluation_random.json','r') as f:
  evaluation_random = json.load(f)

In [ ]:
container_for_id_score = []
for i in range(len(evaluation_high['ID'])):
  container_for_id_score.append((evaluation_high['ID'][i], evaluation_high['has_Overall_Reliability'][i], i))

In [ ]:
print(container_for_id_score)

[(595, 100, 0), (17, 95, 1), (374, 95, 2), (948, 95, 3), (710, 95, 4), (388, 95, 5), (908, 100, 6), (386, 95, 7), (604, 95, 8), (296, 40, 9), (848, 100, 10), (120, 95, 11), (390, 100, 12), (583, None, 13), (918, 95, 14), (713, 100, 15), (9, 100, 16), (945, 95, 17), (303, 95, 18), (149, 95, 19), (76, 100, 20), (224, 95, 21), (228, 100, 22), (298, 95, 23), (199, 100, 24), (498, 95, 25), (886, 95, 26), (136, 95, 27), (725, 95, 28), (673, 95, 29), (458, 95, 30), (70, 95, 31), (791, None, 32), (649, 95, 33), (596, 95, 34), (442, 95, 35), (861, 95, 36), (825, None, 37), (123, None, 38), (271, 95, 39), (181, None, 40), (783, 95, 41), (278, 95, 42), (516, 95, 43), (126, 95, 44), (171, None, 45), (921, 95, 46), (300, None, 47), (781, 95, 48), (479, 95, 49), (913, 95, 50), (161, 95, 51), (920, 100, 52), (560, 95, 53), (422, 75, 54), (217, 95, 55), (582, 100, 56), (685, 100, 57), (446, 98, 58), (584, 95, 59), (213, 95, 60), (707, None, 61), (664, 95, 62), (450, 95, 63), (401, 95, 64), (10, None, 

In [ ]:
# data (dict) -> function -> extract datas : has_halucination = 1, has_contradiction = 1, has_irrelevant = 1, has_overall_reliability = None

def data_filter_for_eval(data):
  # data = [dict]
  halucination = []
  contradiction = []
  irrelevant = []
  overall_reliability = []
  for  i, id in enumerate(data['ID']):
    if data['has_Halucination'][i] == 1:
      halu = (id ,data['has_Halucination'][i], i)
      halucination.append(halu)
    if data['has_Contradiction'][i] == 1:
      contra = (id, data['has_Contradiction'][i], i)
      contradiction.append(contra)
    if data['has_Irrelevant'][i] == 1:
      irrel = (id, data['has_Irrelevant'][i], i)
      irrelevant.append(irrel)
    if type(data['has_Overall_Reliability'][i]) != int:
      reliability = (id, data['has_Overall_Reliability'][i], i)
      overall_reliability.append(reliability)
  print('Haluciantion: num :{} \n {}'.format(len(halucination), halucination))
  print('Contradiction: num :{} \n {}'.format(len(contradiction) , contradiction))
  print('irrelevant: num :{} \n {}'.format(len(irrelevant) , irrelevant))
  print('Overall_reliability_score: num :{} \n {}'.format(len(overall_reliability) , overall_reliability))

  return halucination, contradiction, irrelevant, overall_reliability


In [ ]:
def fix_overall_score(data,data_list):
  for i in data_list:
    print(data['Evaluation'][i[2]])
    score = input('score: ')
    data['has_Overall_Reliability'][i[2]] = int(score)

In [ ]:
def data_info(data):
  average = 0
  count_for_under_50 = 0
  for i in data['has_Overall_Reliability']:
    average += i
    if i <= 50:
      count_for_under_50 += 1
  average = average / len(data['has_Overall_Reliability'])
  minimum = min(data['has_Overall_Reliability'])
  maximum = max(data['has_Overall_Reliability'])
  print('Group: {}'.format(data['Group']))
  print('Under Score 50: {}'.format(count_for_under_50))
  print('Average:{}'.format(average))
  print('MInimum Score: {}'.format(minimum))
  print('Maximum Score: {}'.format(maximum))

In [ ]:
halucination_high, contradiction_high, irrelevant_high, overall_reliability_high = data_filter_for_eval(evaluation_high)
print('----------------------------')
halucination_low, contradiction_low, irrelevant_low, overall_reliability_low = data_filter_for_eval(evaluation_low)
print('----------------------------')
halucination_random, contradiction_random, irrelevant_random, overall_reliability_random = data_filter_for_eval(evaluation_random)
print('----------------------------')

Haluciantion: num :3 
 [(296, 1, 9), (583, 1, 13), (772, 1, 82)]
Contradiction: num :4 
 [(296, 1, 9), (123, 1, 38), (422, 1, 54), (772, 1, 82)]
irrelevant: num :0 
 []
Overall_reliability_score: num :0 
 []
----------------------------
Haluciantion: num :1 
 [(592, 1, 45)]
Contradiction: num :0 
 []
irrelevant: num :0 
 []
Overall_reliability_score: num :9 
 [(884, None, 1), (114, None, 14), (196, None, 17), (27, None, 38), (819, None, 59), (555, None, 61), (349, None, 71), (840, None, 72), (752, None, 81)]
----------------------------
Haluciantion: num :1 
 [(651, 1, 69)]
Contradiction: num :2 
 [(148, 1, 53), (651, 1, 69)]
irrelevant: num :0 
 []
Overall_reliability_score: num :4 
 [(460, None, 15), (324, None, 63), (565, None, 74), (634, None, 94)]
----------------------------


In [ ]:
fix_overall_score(evaluation_low, overall_reliability_low)

Hallucination: No
Contradiction: No
Irrelevant: No
Overall Reliability Score (95): The summary accurately reflects the main points and concerns raised in the original article regarding the trend of athletes hiring personal trainers and the associated risks and challenges for teams. It successfully captures the central debate and the differing perspectives of trainers and club officials.
score: 95
Hallucination: No
Contradiction: No
Irrelevant: No
Overall Reliability Score (95): The summary accurately reflects the key information presented in the original article regarding the takeover deal, the CVA process, and Green’s intentions. It successfully captures the core elements without introducing any extraneous details or inconsistencies.
score: 95
Hallucination: No
Contradiction: No
Irrelevant: No
Overall Reliability Score (95): The summary accurately reflects the core events and outcomes described in the original article, presenting a balanced overview without introducing any fabricated 

In [ ]:
data_info(evaluation_low)

Group: Low
Under Score 50: 0
Average:94.83333333333333
MInimum Score: 65
Maximum Score: 100


In [ ]:
fix_overall_score(evaluation_random, overall_reliability_random)

Hallucination: No
Contradiction: No
Irrelevant: No
Overall Reliability Score (95): The summary accurately reflects the main points and concerns discussed in the article regarding the Johnson & Johnson vaccine’s review and potential limitations. It avoids introducing any fabricated information or conflicting statements.
score: 95
Hallucination: No
Contradiction: No
Irrelevant: No
Overall Reliability Score (95): The summary accurately reflects the key facts and arguments presented in the original article, avoiding any fabricated information or contradictions. It effectively captures the core narrative of Biden’s strong job growth and the factors contributing to it, alongside the public’s negative perception due to inflation.
score: 95
Hallucination: No
Contradiction: No
Irrelevant: No
Overall Reliability Score (95): This summary accurately reflects the key events and details presented in the original article. It successfully captures Djokovic’s return and Federer’s family news while main

In [ ]:
data_info(evaluation_random)

Group: Random
Under Score 50: 1
Average:94.5625
MInimum Score: 30
Maximum Score: 100


In [ ]:
data_info(evaluation_high)

Group: High
Under Score 50: 1
Average:94.29166666666667
MInimum Score: 40
Maximum Score: 100


In [ ]:
#save overall score.
with open('evaluation_high.json','w') as f:
  json.dump(evaluation_high, f)

with open('evaluation_low.json','w') as f:
  json.dump(evaluation_low, f)

with open('evaluation_random.json','w') as f:
  json.dump(evaluation_random, f)

In [ ]:
#data has quite good scores in general so it's reliable

In [ ]:
with open('data_model_generated.json', 'r') as f:
  preprocessd_data = json.load(f)

In [ ]:
data_for_training = {
    'original_article': preprocessed_data['original_article'],
    'model_output': preprocessed_data['model_output'],
    'subject' :  preprocessed_data['subject'],
    'keywords' : preprocessed_data['keywords'],
    'summary' : preprocessed_data['summary']
}

In [ ]:
with open('dataset_for_training.json', 'w') as f:
  json.dump(data_for_training, f)

In [ ]:
with open('dataset_for_training.json', 'r') as f:
  data = json.load(f)